In [ ]:
import re
import json
import pdfplumber

pdf_path = "../input_files/BD01529397W_workorder.pdf"

extracted_texts = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                for cell in row:
                    if cell and cell.strip():
                        extracted_texts.append(cell.strip())

extracted_texts

['WORKS ORDER: DUPLICATE',
 "Chain: VICTORIA'S SECRET",
 'Order Header\nDetails:',
 'Works Order BD01529397W Order Dihan Ahmed Bangladesh |\nConfirmed\nNo: EMail: dihan.ahmed@itl-group.com.bd\nBy:\nTo: INTERNATIONAL TRIMMINGS & LABELS BANGLADESH PRIVATE LIMITED.\nCustomer: Mas Intimates Bangladesh Private Limited [M096]\nCustomer Order No: 4502859819-1001667399/10-2001431649-200',
 'ITL BD PRODUCTION SPECIFICATIONS:',
 'ITL BD Material Code:\nFinishing Material\nCode:\nFinishing Information:\nProduction\nDepartment:\nProduction to Follow: Follow approved card of LB 07661 C/1 for PINK Gossip color, others as per approved AW.\nSpecial Note:',
 'Order Delivery Details:',
 'PO Received Date: 2026/06/03\nOrder Date: 2026/06/03\nDelivery Date (ex\n2026/07/30\nfactory):\nCustomer Delivery\nMas Intimates Bangladesh Private Limited\nName:\nMas Intimates Bangladesh Private Limited,SFB No. 1,KEPZ, North Patenga, Chittagong-4204, ,\nDeliver To:\nBangladesh\nDelivery Method: ITL\nDelivery Account N

In [14]:
def split_work_orders(extracted_texts):
    work_orders = []

    start_pos = 0
    i = 0

    while i < len(extracted_texts) - 1:
        current_item = extracted_texts[i]
        next_item = extracted_texts[i + 1]

        if "End of Works Order:" not in current_item:
            i += 1
            continue

        work_order_match = re.search(r"\*([^*]+)\*", next_item)

        if not work_order_match:
            i += 1
            continue

        work_order_no = work_order_match.group(1).strip()
        block_items = extracted_texts[start_pos:i + 2]
        block_text = "\n".join(block_items)

        work_order_exists_in_block = work_order_no in block_text

        work_orders.append({
            "work_order_no": work_order_no,
            "work_order_exists_in_block": work_order_exists_in_block,
            "items": block_items,
        })

        start_pos = i + 2
        i = start_pos

    return work_orders

In [15]:
split_work_orders(extracted_texts)

[{'work_order_no': 'BD01529397W',
  'work_order_exists_in_block': True,
  'items': ['WORKS ORDER: DUPLICATE',
   "Chain: VICTORIA'S SECRET",
   'Order Header\nDetails:',
   'Works Order BD01529397W Order Dihan Ahmed Bangladesh |\nConfirmed\nNo: EMail: dihan.ahmed@itl-group.com.bd\nBy:\nTo: INTERNATIONAL TRIMMINGS & LABELS BANGLADESH PRIVATE LIMITED.\nCustomer: Mas Intimates Bangladesh Private Limited [M096]\nCustomer Order No: 4502859819-1001667399/10-2001431649-200',
   'ITL BD PRODUCTION SPECIFICATIONS:',
   'ITL BD Material Code:\nFinishing Material\nCode:\nFinishing Information:\nProduction\nDepartment:\nProduction to Follow: Follow approved card of LB 07661 C/1 for PINK Gossip color, others as per approved AW.\nSpecial Note:',
   'Order Delivery Details:',
   'PO Received Date: 2026/06/03\nOrder Date: 2026/06/03\nDelivery Date (ex\n2026/07/30\nfactory):\nCustomer Delivery\nMas Intimates Bangladesh Private Limited\nName:\nMas Intimates Bangladesh Private Limited,SFB No. 1,KEPZ, Nor

In [16]:
def clean_value(value):
    return " ".join(value.split()).strip()

def find_with_patterns(text, patterns):
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.MULTILINE | re.DOTALL)
        if match:
            return match

    return None

def sequential_extract(items, rules):
    result = {}

    cursor_item = 0
    cursor_char = 0

    for rule in rules:
        key = rule["key"]
        patterns = rule["patterns"]

        found = False

        for item_index in range(cursor_item, len(items)):
            item = items[item_index]

            if item_index == cursor_item:
                start_char = cursor_char
            else:
                start_char = 0

            search_text = item[start_char:]
            match = find_with_patterns(search_text, patterns)

            if not match:
                continue

            result[key] = clean_value(match.group(1))

            cursor_item = item_index
            cursor_char = start_char + match.end()

            found = True
            break

        if not found:
            result[key] = ""

    return result

In [40]:
rules = [
    {
    "key": "customer",
    "patterns": [
        r"Customer:\s*(.*?)\s*\[[A-Z0-9]+\]", # \nCustomer: Mas Intimates Bangladesh Private Limited [M096]\n
        r"Customer:\s*(.+)",
        ],
    },
    {
        "key": "customer_order_no",
        "patterns": [
            r"Customer Order No:\s*([0-9\/\-]+)",
        ],
    },
    {
        "key": "vs_po_number",
        "patterns": [
            r"VS PO Number:\s*([0-9]{10})", # always 10 digits
        ],
    },
    {
        "key": "line_item",
        "patterns": [
            r"Line Item:\s*([0-9]+)",
        ],
    },
    {
        "key": "so_number",
        "patterns": [
            r"SO Number:\s*(\d{10}/\d{2})", # 10 digits, followed by a slash and 2 digits
        ],
    },
    {
        "key": "item_code",
        "patterns": [
            r"Item Code:\s*([0-9]{10})", # always 10 digits
        ],
    },
    {
        "key": "product_code",
        "patterns": [
            r"Product Code:\s*([A-Z0-9-/ ]+)",
        ],
    },
    {
        "key": "silhouette",
        "patterns": [
            r"Silhouette:\s*([A-Z/]+)",
        ],
    },
    {
        "key": "quantity",
        "patterns": [
            r"Quantity:\s*([0-9]+)\sunits", # followed by "units"
        ],
    },
    {
        "key": "size_id",
        "patterns": [
            r"Size ID:\s*(.*?)(?:\n|Size/Age Breakdown:|$)", # non-greedy match until a newline, "Size/Age Breakdown:", or end of string
        ],
    },
    {
        "key": "itl_factory_code",
        "patterns": [
            r"ITL Factory Code:\s*([A-Z][0-9-]+)", # starts with a capital letter followed by digits and/or hyphens
        ],
    },
    {
        "key": "vsd",
        "patterns": [
            r"VSD#:\s*(\d{6}-[A-Z]{3})", # 6 digits followed by a hyphen and 3 capital letters
        ],
    },
    {
        "key": "vss",
        "patterns": [
            r"VSS#:\s*(\d{8})", # always 8 digits
        ],
    },
    {
        "key": "rn",
        "patterns": [
            r"RN#:\s*(\d+)",
        ],
    },
    {
        "key": "ca",
        "patterns": [
            r"CA#:\s*(\d+)",
        ],
    },
    {
        "key": "factory_id",
        "patterns": [
            r"Factory ID:\s*(\d+)",
        ],
    },
    {
        "key": "date_of_mfr",
        "patterns": [
            r"Date of MFR#:\s*(\d{2}\s+\d{2})",
        ],
    },
    {
        "key": "country_of_origin",
        "patterns": [
            r"Country Of Origin\s*(.*?)(?:\n|Additional Instructions:|$)",
        ],
    },
    {
        "key": "additional_instructions",
        "patterns": [
            r"Additional Instructions:\s*(.*?)(?:\n|Garment Components|$)",
        ],
    },
    {
        "key": "garment_components",
        "patterns": [
            r"Garment Components\s*(.*?)(?:\nCare Instructions:|\Z)",
        ],
    },
    {
        "key": "care_instructions_set_1",
        "patterns": [
            r"Care Instruction Set 1:\s*([A-Z0-9]+)",
        ],
    }
]

In [41]:
work_order_blocks = split_work_orders(extracted_texts)

all_work_orders = []

for block in work_order_blocks:
    result = {
        "work_order_no": block["work_order_no"],
        "work_order_exists_in_block": block["work_order_exists_in_block"],
    }

    if block["work_order_exists_in_block"]:
        extracted_values = sequential_extract(block["items"], rules)
        result.update(extracted_values)
    else:
        result["error"] = "Work order number not found inside this block"

    all_work_orders.append(result)

print(json.dumps(all_work_orders, indent=4, ensure_ascii=False))

[
    {
        "work_order_no": "BD01529397W",
        "work_order_exists_in_block": true,
        "customer": "Mas Intimates Bangladesh Private Limited",
        "customer_order_no": "4502859819-1001667399/10-2001431649-200",
        "vs_po_number": "4502859819",
        "line_item": "200",
        "so_number": "1001667399/10",
        "item_code": "2001431649",
        "product_code": "LB 07655 C/1 / PPK-1023",
        "silhouette": "SHORTIE/SHORTIE",
        "quantity": "3066",
        "size_id": "VSGLOBAL003 - Panties/Swim Bottoms",
        "itl_factory_code": "B1-385",
        "vsd": "416213-SPT",
        "vss": "",
        "rn": "54867",
        "ca": "67359",
        "factory_id": "36014990",
        "date_of_mfr": "07 26",
        "country_of_origin": "made in Bangladesh/fabriqué au Bangladesh/hecho en Bangladesh/fabbricato in Bangladesh/孟加拉制造",
        "additional_instructions": "exclusive of decoration/sauf décoration/no incluye la decoración/esclusa la decorazione/装饰除外",
  

'VSGLOB03 - Panties/Swim Bottoms'

## Size/Age Breakdown Extraction

In [58]:
import pdfplumber

pdf_path = "../input_files/BD01529397W_workorder.pdf"

def clean_cell(value):
    return " ".join((value or "").split()).strip()

def is_number(value):
    value = clean_cell(value).replace(",", "")
    return value.isdigit()

def extract_size_age_breakdown(pdf_path):
    size_rows = []
    header = None
    collecting = False

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()

            for table in tables:
                for row in table:
                    cells = [clean_cell(cell) for cell in row]

                    if not any(cells):
                        continue

                    # Find the Size/Age Breakdown header row
                    if not collecting:
                        if cells[-1] == "Order Quantity":
                            header = cells
                            collecting = True
                        continue

                    # Skip repeated header if table continues on next page
                    if cells == header:
                        continue

                    # Data row length must match header length
                    if len(cells) != len(header):
                        continue

                    # Last column must be numeric quantity
                    if not is_number(cells[-1]):
                        continue

                    row_data = dict(zip(header, cells))
                    size_rows.append(row_data)

    return size_rows

size_age_breakdown = extract_size_age_breakdown(pdf_path)

size_age_breakdown

[{'Panties/Swim Bottoms': 'L | G | 170/80A',
  'Line No': '',
  'Order Quantity': '404'},
 {'Panties/Swim Bottoms': 'M | 170/72A',
  'Line No': '',
  'Order Quantity': '1287'},
 {'Panties/Swim Bottoms': 'S | P | CH | 170/68A',
  'Line No': '',
  'Order Quantity': '718'},
 {'Panties/Swim Bottoms': 'XL | XG | EG | 170/90A',
  'Line No': '',
  'Order Quantity': '428'},
 {'Panties/Swim Bottoms': 'XS | XP | ECH | 165/64A',
  'Line No': '',
  'Order Quantity': '229'}]

## Merging 2 approaches: Extracting structured data with regex patterns, and extracting the size/age breakdown table separately. 

In [60]:
import re
import json
import pdfplumber

pdf_path = "../input_files/BD01529397W_workorder.pdf"

def clean_cell(value):
    return " ".join((value or "").split()).strip()

def is_number(value):
    return clean_cell(value).replace(",", "").isdigit()

def extract_pdf_once(pdf_path):
    extracted_texts = []
    size_age_rows = []

    size_header = None
    collecting_size = False

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()

            for table in tables:
                for row in table:
                    row_start_pos = len(extracted_texts)

                    cells = [clean_cell(cell) for cell in row]

                    # Plain text extraction
                    for cell in cells:
                        if cell:
                            extracted_texts.append(cell)

                    if not any(cells):
                        continue

                    # Find dynamic Size/Age Breakdown header
                    if cells[-1] == "Order Quantity":
                        size_header = cells
                        collecting_size = True
                        continue

                    if not collecting_size or size_header is None:
                        continue

                    # Skip repeated header on next page
                    if cells == size_header:
                        continue

                    # Row length must match header length
                    if len(cells) != len(size_header):
                        continue

                    # Last value must be numeric quantity
                    if not is_number(cells[-1]):
                        continue

                    size_age_rows.append({
                        "position": row_start_pos,
                        "data": dict(zip(size_header, cells)),
                    })

    return extracted_texts, size_age_rows

In [61]:
def split_work_orders(extracted_texts):
    work_orders = []

    start_pos = 0
    i = 0

    while i < len(extracted_texts) - 1:
        current_item = extracted_texts[i]
        next_item = extracted_texts[i + 1]

        if "End of Works Order:" not in current_item:
            i += 1
            continue

        work_order_match = re.search(r"\*([^*]+)\*", next_item)

        if not work_order_match:
            i += 1
            continue

        work_order_no = work_order_match.group(1).strip()
        block_items = extracted_texts[start_pos:i + 2]
        block_text = "\n".join(block_items)

        work_orders.append({
            "work_order_no": work_order_no,
            "work_order_exists_in_block": work_order_no in block_text,
            "start_pos": start_pos,
            "end_pos": i + 1,
            "items": block_items,
        })

        start_pos = i + 2
        i = start_pos

    return work_orders

In [64]:
extracted_texts, size_age_rows = extract_pdf_once(pdf_path)

work_order_blocks = split_work_orders(extracted_texts)
print(work_order_blocks)

all_work_orders = []

for block in work_order_blocks:
    result = {
        "work_order_no": block["work_order_no"],
        "work_order_exists_in_block": block["work_order_exists_in_block"],
    }

    if block["work_order_exists_in_block"]:
        extracted_values = sequential_extract(block["items"], rules)
        result.update(extracted_values)

        size_rows_for_this_work_order = [
            row["data"]
            for row in size_age_rows
            if block["start_pos"] <= row["position"] <= block["end_pos"]
        ]

        result["size_age_breakdown"] = size_rows_for_this_work_order

    else:
        result["error"] = "Work order number not found inside this block"

    all_work_orders.append(result)

print(json.dumps(all_work_orders, indent=4, ensure_ascii=False))

[{'work_order_no': 'BD01529397W', 'work_order_exists_in_block': True, 'start_pos': 0, 'end_pos': 38, 'items': ['WORKS ORDER: DUPLICATE', "Chain: VICTORIA'S SECRET", 'Order Header Details:', 'Works Order BD01529397W Order Dihan Ahmed Bangladesh | Confirmed No: EMail: dihan.ahmed@itl-group.com.bd By: To: INTERNATIONAL TRIMMINGS & LABELS BANGLADESH PRIVATE LIMITED. Customer: Mas Intimates Bangladesh Private Limited [M096] Customer Order No: 4502859819-1001667399/10-2001431649-200', 'ITL BD PRODUCTION SPECIFICATIONS:', 'ITL BD Material Code: Finishing Material Code: Finishing Information: Production Department: Production to Follow: Follow approved card of LB 07661 C/1 for PINK Gossip color, others as per approved AW. Special Note:', 'Order Delivery Details:', 'PO Received Date: 2026/06/03 Order Date: 2026/06/03 Delivery Date (ex 2026/07/30 factory): Customer Delivery Mas Intimates Bangladesh Private Limited Name: Mas Intimates Bangladesh Private Limited,SFB No. 1,KEPZ, North Patenga, Chit

In [ ]:
[
    {
        "work_order_no": "BD01529397W",
        "work_order_exists_in_block": true,
        "customer": "Mas Intimates Bangladesh Private Limited",
        "customer_order_no": "4502859819-1001667399/10-2001431649-200",
        "vs_po_number": "4502859819",
        "line_item": "200",
        "so_number": "1001667399/10",
        "item_code": "2001431649",
        "product_code": "LB 07655 C/1 / PPK-1023 P",
        "silhouette": "SHORTIE/SHORTIE",
        "quantity": "3066",
        "size_id": "VSGLOBAL003 - Panties/Swim Bottoms",
        "itl_factory_code": "B1-385",
        "vsd": "416213-SPT",
        "vss": "",
        "rn": "54867",
        "ca": "67359",
        "factory_id": "36014990",
        "date_of_mfr": "07 26",
        "country_of_origin": "made in Bangladesh/fabriqué au Bangladesh/hecho en Bangladesh/fabbricato in Bangladesh/孟加拉制造",
        "additional_instructions": "",
        "garment_components": "body/gusset/corps/gousset/cuerpo/inserto/corpo/tassello/面料/内裆: & 57% cotton/coton/algodón/cotone/棉 Fibre Contents: 38% modal/莫代尔 5% elastane/élasthanne/elastano/elastan/氨纶 100% (Total) elastic/élastique/banda elástica/elastico/松紧带 48% polyamide/poliamida/poliammide/锦纶 37% polyester/poliéster/poliestere/聚酯纤维 15% elastane/élasthanne/elastano/elastan/氨纶 100% (Total) Care Instructions: Care Instruction Set 1: MWW001",
        "care_instructions_set_1": "MWW001",
        "size_age_breakdown": [
            {
                "Panties/Swim Bottoms": "L | G | 170/80A",
                "Line No": "",
                "Order Quantity": "404"
            },
            {
                "Panties/Swim Bottoms": "M | 170/72A",
                "Line No": "",
                "Order Quantity": "1287"
            },
            {
                "Panties/Swim Bottoms": "S | P | CH | 170/68A",
                "Line No": "",
                "Order Quantity": "718"
            },
            {
                "Panties/Swim Bottoms": "XL | XG | EG | 170/90A",
                "Line No": "",
                "Order Quantity": "428"
            },
            {
                "Panties/Swim Bottoms": "XS | XP | ECH | 165/64A",
                "Line No": "",
                "Order Quantity": "229"
            }
        ]
    }
]

# Clear Code

In [1]:
import re
import json
import pdfplumber

pdf_path = "../input_files/BD01529397W_workorder.pdf"

In [2]:
# Helper functions for cleaning and validating data
def clean_cell(value):
    return " ".join((value or "").split()).strip()

def clean_value(value):
    return " ".join(value.split()).strip()

def is_number(value):
    return clean_cell(value).replace(",", "").isdigit()

def clean_garment_components(value):
    value = value.replace("&", "")
    value = value.replace("Fibre Contents:", "")
    value = " ".join(value.split())
    return value.strip()

In [3]:
def extract_pdf_once(pdf_path):
    extracted_texts = []
    size_age_rows = []

    size_header = None
    collecting_size = False

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()

            for table in tables:
                for row in table:
                    row_start_pos = len(extracted_texts)
                    cells = [clean_cell(cell) for cell in row]

                    # 1. Plain text extraction from table cells
                    for cell in cells:
                        if cell:
                            extracted_texts.append(cell)

                    if not any(cells):
                        continue

                    # 2. Detect Size/Age Breakdown header dynamically
                    if cells[-1] == "Order Quantity":
                        size_header = cells
                        collecting_size = True
                        continue

                    if not collecting_size or size_header is None:
                        continue

                    # 3. Skip repeated header if table continues on another page
                    if cells == size_header:
                        continue

                    # 4. Data row must match header length
                    if len(cells) != len(size_header):
                        continue

                    # 5. Last column must be numeric quantity
                    if not is_number(cells[-1]):
                        continue

                    size_age_rows.append({
                        "position": row_start_pos,
                        "data": dict(zip(size_header, cells)),
                    })

    return extracted_texts, size_age_rows

In [4]:
# Split work orders based on "End of Works Order:" and extract relevant data
def split_work_orders(extracted_texts):
    work_orders = []

    start_pos = 0
    i = 0

    while i < len(extracted_texts) - 1:
        current_item = extracted_texts[i]
        next_item = extracted_texts[i + 1]

        if "End of Works Order:" not in current_item:
            i += 1
            continue

        work_order_match = re.search(r"\*([^*]+)\*", next_item)

        if not work_order_match:
            i += 1
            continue

        work_order_no = work_order_match.group(1).strip()
        block_items = extracted_texts[start_pos:i + 2]
        block_text = "\n".join(block_items)

        work_orders.append({
            "work_order_no": work_order_no,
            "work_order_exists_in_block": work_order_no in block_text,
            "start_pos": start_pos,
            "end_pos": i + 1,
            "items": block_items,
        })

        start_pos = i + 2
        i = start_pos

    return work_orders

In [5]:
# Sequentially extract data based on defined rules
def find_with_patterns(text, patterns):
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.MULTILINE | re.DOTALL)
        if match:
            return match

    return None

def sequential_extract(items, rules):
    result = {}

    cursor_item = 0
    cursor_char = 0

    for rule in rules:
        key = rule["key"]
        patterns = rule["patterns"]

        found = False

        for item_index in range(cursor_item, len(items)):
            item = items[item_index]

            if item_index == cursor_item:
                start_char = cursor_char
            else:
                start_char = 0

            search_text = item[start_char:]
            match = find_with_patterns(search_text, patterns)

            if not match:
                continue

            result[key] = clean_value(match.group(1))

            cursor_item = item_index
            cursor_char = start_char + match.end()

            found = True
            break

        if not found:
            result[key] = ""

    return result

In [6]:
rules = [
    {
        "key": "customer",
        "patterns": [
            r"Customer:\s*(.*?)\s*\[[A-Z0-9]+\]",
            r"Customer:\s*(.+)",
        ],
    },
    {
        "key": "customer_order_no",
        "patterns": [
            r"Customer Order No:\s*([0-9/-]+)",
        ],
    },
    {
        "key": "vs_po_number",
        "patterns": [
            r"VS PO Number:\s*([0-9]{10})",
        ],
    },
    {
        "key": "line_item",
        "patterns": [
            r"Line Item:\s*([0-9]+)",
        ],
    },
    {
        "key": "so_number",
        "patterns": [
            r"SO Number:\s*(\d{10}/\d{2})",
        ],
    },
    {
        "key": "item_code",
        "patterns": [
            r"Item Code:\s*([0-9]{10})",
        ],
    },
    {
        "key": "product_code",
        "patterns": [
            r"Product Code:\s*(.*?)(?=\s*Product Description:|\Z)",
        ],
    },
    {
        "key": "silhouette",
        "patterns": [
            r"Silhouette:\s*([A-Za-z/]+)",
        ],
    },
    {
        "key": "quantity",
        "patterns": [
            r"Quantity:\s*([0-9]+)\s+units",
        ],
    },
    {
        "key": "size_id",
        "patterns": [
            r"Size ID:\s*(.*?)(?:\n|Size/Age Breakdown:|\Z)",
        ],
    },
    {
        "key": "itl_factory_code",
        "patterns": [
            r"ITL Factory Code:\s*([A-Z][0-9-]+)",
        ],
    },
    {
        "key": "vsd",
        "patterns": [
            r"VSD#:\s*(\d{6}-[A-Z]{3})",
        ],
    },
    {
        "key": "vss",
        "patterns": [
            r"VSS#:\s*(\d{8})",
        ],
    },
    {
        "key": "rn",
        "patterns": [
            r"RN#:\s*(\d+)",
        ],
    },
    {
        "key": "ca",
        "patterns": [
            r"CA#:\s*(\d+)",
        ],
    },
    {
        "key": "factory_id",
        "patterns": [
            r"Factory ID:\s*(\d+)",
        ],
    },
    {
        "key": "date_of_mfr",
        "patterns": [
            r"Date of MFR#:\s*(\d{2}\s+\d{2})",
        ],
    },
    {
        "key": "country_of_origin",
        "patterns": [
            r"Country Of Origin\s*(.*?)(?=\s*Additional Instructions:|\Z)",
        ],
    },
    {
        "key": "additional_instructions",
        "patterns": [
            r"Additional Instructions:\s*(.*?)(?=\s*Garment Components|\Z)",
        ],
    },
    {
        "key": "garment_components",
        "patterns": [
            r"Garment Components\s*(.*?)(?=\s*Care Instructions:|\Z)",
        ],
    },
    {
        "key": "care_instructions_set_1",
        "patterns": [
            r"Care Instruction Set 1:\s*([A-Z0-9]+)",
        ],
    },
]

In [7]:
# Run full extraction process
extracted_texts, size_age_rows = extract_pdf_once(pdf_path)

work_order_blocks = split_work_orders(extracted_texts)

all_work_orders = []

for block in work_order_blocks:
    result = {
        "work_order_no": block["work_order_no"],
        "work_order_exists_in_block": block["work_order_exists_in_block"],
    }

    if block["work_order_exists_in_block"]:
        extracted_values = sequential_extract(block["items"], rules)

        if extracted_values.get("garment_components"):
            extracted_values["garment_components"] = clean_garment_components(
                extracted_values["garment_components"]
            )

        result.update(extracted_values)

        result["size_age_breakdown"] = [
            row["data"]
            for row in size_age_rows
            if block["start_pos"] <= row["position"] <= block["end_pos"]
        ]

    else:
        result["error"] = "Work order number not found inside this block"

    all_work_orders.append(result)

print(json.dumps(all_work_orders, indent=4, ensure_ascii=False))

[
    {
        "work_order_no": "BD01529397W",
        "work_order_exists_in_block": true,
        "customer": "Mas Intimates Bangladesh Private Limited",
        "customer_order_no": "4502859819-1001667399/10-2001431649-200",
        "vs_po_number": "4502859819",
        "line_item": "200",
        "so_number": "1001667399/10",
        "item_code": "2001431649",
        "product_code": "LB 07655 C/1 / PPK-1023",
        "silhouette": "SHORTIE/SHORTIE",
        "quantity": "3066",
        "size_id": "VSGLOBAL003 - Panties/Swim Bottoms",
        "itl_factory_code": "B1-385",
        "vsd": "416213-SPT",
        "vss": "",
        "rn": "54867",
        "ca": "67359",
        "factory_id": "36014990",
        "date_of_mfr": "07 26",
        "country_of_origin": "made in Bangladesh/fabriqué au Bangladesh/hecho en Bangladesh/fabbricato in Bangladesh/孟加拉制造",
        "additional_instructions": "exclusive of decoration/sauf décoration/no incluye la decoración/esclusa la decorazione/装饰除外",
  

In [ ]:
[
    {
        "work_order_no": "BD01529397W",
        "work_order_exists_in_block": true,
        "customer": "Mas Intimates Bangladesh Private Limited",
        "customer_order_no": "4502859819-1001667399/10-2001431649-200",
        "vs_po_number": "4502859819",
        "line_item": "200",
        "so_number": "1001667399/10",
        "item_code": "2001431649",
        "product_code": "LB 07655 C/1 / PPK-1023",
        "silhouette": "SHORTIE/SHORTIE",
        "quantity": "3066",
        "size_id": "VSGLOBAL003 - Panties/Swim Bottoms",
        "itl_factory_code": "B1-385",
        "vsd": "416213-SPT",
        "vss": "",
        "rn": "54867",
        "ca": "67359",
        "factory_id": "36014990",
        "date_of_mfr": "07 26",
        "country_of_origin": "made in Bangladesh/fabriqué au Bangladesh/hecho en Bangladesh/fabbricato in Bangladesh/孟加拉制造",
        "additional_instructions": "exclusive of decoration/sauf décoration/no incluye la decoración/esclusa la decorazione/装饰除外",
        "garment_components": "body/gusset/corps/gousset/cuerpo/inserto/corpo/tassello/面料/内裆: 57% cotton/coton/algodón/cotone/棉 38% modal/莫代尔 5% elastane/élasthanne/elastano/elastan/氨纶 100% (Total) elastic/élastique/banda elástica/elastico/松紧带 48% polyamide/poliamida/poliammide/锦纶 37% polyester/poliéster/poliestere/聚酯纤维 15% elastane/élasthanne/elastano/elastan/氨纶 100% (Total)",
        "care_instructions_set_1": "MWW001",
        "size_age_breakdown": [
            {
                "Panties/Swim Bottoms": "L | G | 170/80A",
                "Line No": "",
                "Order Quantity": "404"
            },
            {
                "Panties/Swim Bottoms": "M | 170/72A",
                "Line No": "",
                "Order Quantity": "1287"
            },
            {
                "Panties/Swim Bottoms": "S | P | CH | 170/68A",
                "Line No": "",
                "Order Quantity": "718"
            },
            {
                "Panties/Swim Bottoms": "XL | XG | EG | 170/90A",
                "Line No": "",
                "Order Quantity": "428"
            },
            {
                "Panties/Swim Bottoms": "XS | XP | ECH | 165/64A",
                "Line No": "",
                "Order Quantity": "229"
            }
        ]
    }
]